# Milestone 2 — R7 Model Building: Hypothesis 1

**Research question:** Does home EV charging improve prediction of overnight grid demand after accounting for temperature, month, and hour?

**R7 rubric alignment:** (1) justify suitable candidate models, (2) compare them on the same observations and chronological split while varying only the charging feature, and (3) log every run with parameters, metrics, and artifacts in MLflow for reproducibility. The early cells preserve the R6 experiment and prepare its data; the later cells implement R7.

**Important:** This notebook covers Hypothesis 1 only. Its recorded outputs must be rerun with the real source CSVs; Hypothesis 2 needs its own model comparison.


### Load the source datasets

Read the charging sessions, hourly grid/weather records, and flagged battery-capacity sessions. Remove matching flagged IDs before analysis. This establishes the input data used by both candidate models; the CSV files must be available locally.


In [1]:
import pandas as pd
from pathlib import Path

charging_path = Path("home_charging_clean.csv")
grid_path = Path("combined_hourly_clean.csv")

# Load the full home charging dataset
sessions = pd.read_csv(
    charging_path,
    usecols=["session_id", "start_time", "end_time", "kwh", "home_or_public"],
    parse_dates=["start_time", "end_time"]
)

# Load hourly grid and weather data
grid = pd.read_csv(
    grid_path,
    usecols=["local_time", "grid_utc", "demand_mwh", "temp_f"],
    parse_dates=["local_time", "grid_utc"]
)

# Load the CSV containing flagged sessions
flagged = pd.read_csv("battery_capacity_issues.csv")

# Identify matching session IDs
flagged_ids = set(flagged["session_id"])
matches = sessions["session_id"].isin(flagged_ids)

print("Flagged IDs:", len(flagged_ids))
print("Matching sessions:", matches.sum())
print("Unmatched flagged IDs:", len(flagged_ids - set(sessions.loc[matches, "session_id"])))
print("Duplicate session IDs:", sessions["session_id"].duplicated().sum())

# Exclude flagged sessions
sessions_clean = sessions.loc[~matches].copy()

print("Sessions remaining:", len(sessions_clean))



Flagged IDs: 20
Matching sessions: 20
Unmatched flagged IDs: 0
Duplicate session IDs: 0
Sessions remaining: 2447974


### Check session durations and daylight-saving dates

Identify sessions with nonpositive durations and remove those outside daylight-saving transition dates. Transition-date records are retained temporarily for a separate ambiguity check rather than silently treated as ordinary timestamps.


In [2]:
# Identify the four daylight-saving transition dates
dst_dates = pd.to_datetime([
    "2022-03-13",
    "2022-11-06",
    "2023-03-12",
    "2023-11-05"
])

# Identify sessions with invalid durations
invalid_duration = (
    sessions_clean["end_time"] <= sessions_clean["start_time"]
)

# Identify sessions occurring on DST transition dates
on_dst_date = (
    sessions_clean["start_time"].dt.normalize().isin(dst_dates)
    | sessions_clean["end_time"].dt.normalize().isin(dst_dates)
)

# Exclude invalid sessions outside DST dates
remove_invalid = invalid_duration & ~on_dst_date

sessions_valid = sessions_clean.loc[~remove_invalid].copy()

# Verify the result
print("Invalid sessions excluded:", remove_invalid.sum())
print("Sessions remaining:", len(sessions_valid))

Invalid sessions excluded: 21
Sessions remaining: 2447953


### Demonstrate hourly charging allocation

Use a small example to show how a charging session spanning multiple clock hours is divided proportionally by time. The allocated energy should sum to the original session energy; this is a sanity check before applying the calculation to real sessions.


In [3]:
from datetime import timedelta

# Example charging session
start = pd.Timestamp("2022-09-01 20:30:00")
end = pd.Timestamp("2022-09-01 22:00:00")
energy = 9.0

# Calculate duration and average charging power
duration = (end - start).total_seconds() / 3600
avg_power = energy / duration

print("Duration (hours):", duration)
print("Average power (kW):", avg_power)

# Find the beginning of the first hour
current_hour = start.floor("h")

# Store the hourly allocations
hourly_energy = []

while current_hour < end:
    next_hour = current_hour + timedelta(hours=1)

    # Determine the portion of the session within this hour
    overlap_start = max(start, current_hour)
    overlap_end = min(end, next_hour)

    overlap_hours = (
        overlap_end - overlap_start
    ).total_seconds() / 3600

    # Allocate energy based on time spent charging
    allocated_kwh = avg_power * overlap_hours

    hourly_energy.append({
        "hour": current_hour,
        "kwh": allocated_kwh
    })

    current_hour = next_hour

display(pd.DataFrame(hourly_energy))

print("Original energy:", energy)
print("Allocated energy:", sum(row["kwh"] for row in hourly_energy))

Duration (hours): 1.5
Average power (kW): 6.0


,hour,kwh
0,2022-09-01 20:00:00,3.0
1,2022-09-01 21:00:00,6.0


Original energy: 9.0
Allocated energy: 9.0


### Define the reusable hourly allocation function

Turn the worked example into `allocate_hourly`, which distributes each valid session’s kWh across overlapping hours. The example at the end checks that the function conserves the supplied energy.


In [4]:
def allocate_hourly(start, end, energy):
    # Calculate session duration in hours
    duration = (end - start).total_seconds() / 3600

    if duration <= 0:
        raise ValueError("Session duration must be positive")

    # Calculate average charging power
    avg_power = energy / duration

    # Begin at the first overlapping hour
    current_hour = start.floor("h")
    hourly_energy = []

    while current_hour < end:
        next_hour = current_hour + pd.Timedelta(hours=1)

        overlap_start = max(start, current_hour)
        overlap_end = min(end, next_hour)

        overlap_hours = (
            overlap_end - overlap_start
        ).total_seconds() / 3600

        allocated_kwh = avg_power * overlap_hours

        hourly_energy.append({
            "hour": current_hour,
            "kwh": allocated_kwh
        })

        current_hour = next_hour

    return hourly_energy

test_result = allocate_hourly(
    pd.Timestamp("2022-09-01 20:30:00"),
    pd.Timestamp("2022-09-01 22:00:00"),
    9.0
)

display(pd.DataFrame(test_result))

print(
    "Total allocated:",
    sum(row["kwh"] for row in test_result)
)

,hour,kwh
0,2022-09-01 20:00:00,3.0
1,2022-09-01 21:00:00,6.0


Total allocated: 9.0


### Identify daylight-saving ambiguity

Flag sessions overlapping the specified daylight-saving transition windows, including invalid-duration records on those dates. Excluding ambiguous local timestamps reduces the risk of assigning charging to the wrong grid hour.


In [5]:
# Define daylight-saving transition windows
dst_windows = [
    ("2022-03-13", "02:00", "03:00"),
    ("2022-11-06", "01:00", "02:00"),
    ("2023-03-12", "02:00", "03:00"),
    ("2023-11-05", "01:00", "02:00")
]

# Initially, no sessions are marked as ambiguous
dst_ambiguous = pd.Series(False, index=sessions_valid.index)

for date, start_hour, end_hour in dst_windows:
    window_start = pd.Timestamp(f"{date} {start_hour}")
    window_end = pd.Timestamp(f"{date} {end_hour}")

    # Identify sessions overlapping the transition window
    affected = (
        (sessions_valid["start_time"] < window_end) &
        (sessions_valid["end_time"] > window_start)
    )

    # Include invalid-duration sessions on transition dates
    invalid_on_date = (
        (sessions_valid["end_time"] <= sessions_valid["start_time"]) &
        (
            (sessions_valid["start_time"].dt.normalize() == pd.Timestamp(date)) |
            (sessions_valid["end_time"].dt.normalize() == pd.Timestamp(date))
        )
    )

    dst_ambiguous = dst_ambiguous | affected | invalid_on_date

print("DST-ambiguous sessions:", dst_ambiguous.sum())
# Exclude sessions flagged as DST-ambiguous
sessions_ready = sessions_valid.loc[~dst_ambiguous].copy()

print("DST-ambiguous sessions excluded:", dst_ambiguous.sum())
print("Sessions remaining:", len(sessions_ready))

# Verify that no flagged sessions remain
print(
    "Flagged sessions remaining:",
    sessions_ready.index.isin(
        sessions_valid.index[dst_ambiguous]
    ).sum()
)

DST-ambiguous sessions: 2873
DST-ambiguous sessions excluded: 2873
Sessions remaining: 2445080
Flagged sessions remaining: 0


### Finalize eligible charging sessions

Create the final session subset after the daylight-saving checks. This cell repeats the filtering performed above and prints the remaining count for verification.


### Aggregate charging energy by hour

Apply the allocation function to every eligible session and sum contributions sharing the same local hour. Transition dates are skipped. The resulting hourly totals become the charging feature used in the later model comparison.


In [6]:
# Convert DST dates into a set for faster lookup
dst_dates_set = set(dst_dates)

# Store total charging energy for each hour
hourly_totals = {}

# Track progress
total_sessions = len(sessions_ready)

for i, session in enumerate(sessions_ready.itertuples(index=False), start=1):

    # Allocate each session's energy across overlapping hours
    allocated_hours = allocate_hourly(
        session.start_time,
        session.end_time,
        session.kwh
    )

    for allocation in allocated_hours:
        hour = allocation["hour"]

        # Exclude daylight saving transition dates
        if hour.normalize() in dst_dates_set:
            continue

        # Accumulate charging energy for each hour
        hourly_totals[hour] = (
            hourly_totals.get(hour, 0.0) + allocation["kwh"]
        )

    # Display progress every 250,000 sessions
    if i % 250000 == 0:
        print(f"Processed {i:,} / {total_sessions:,} sessions")

print("Unique hours:", len(hourly_totals))
print("Total allocated energy:", sum(hourly_totals.values()))

Processed 250,000 / 2,445,080 sessions
Processed 500,000 / 2,445,080 sessions
Processed 750,000 / 2,445,080 sessions
Processed 1,000,000 / 2,445,080 sessions
Processed 1,250,000 / 2,445,080 sessions
Processed 1,500,000 / 2,445,080 sessions
Processed 1,750,000 / 2,445,080 sessions
Processed 2,000,000 / 2,445,080 sessions
Processed 2,250,000 / 2,445,080 sessions
Unique hours: 11566
Total allocated energy: 32566999.861260578


### Create the hourly charging feature table

Convert the aggregated hourly totals into a sorted table with `local_time` and `home_charging_kwh`. This gives the charging data the same hourly structure needed for the grid-data join.


In [7]:
hourly_homeCharging_demand = pd.DataFrame(list(hourly_totals.items()), columns=["local_time", "home_charging_kwh"])
hourly_homeCharging_demand = hourly_homeCharging_demand.sort_values(by="local_time")

hourly_homeCharging_demand = hourly_homeCharging_demand.reset_index(drop=True)
display(hourly_homeCharging_demand.head())

,local_time,home_charging_kwh
0,2022-09-01 14:00:00,5.287614
1,2022-09-01 15:00:00,3.563183
2,2022-09-01 16:00:00,4.554365
3,2022-09-01 17:00:00,7.773969
4,2022-09-01 18:00:00,25.747746


### Align charging with grid and weather observations

Restrict the grid data to the analysis period, exclude daylight-saving dates, and join charging totals by local hour. Missing joined charging values are set to zero, an assumption that should be checked against the data-collection coverage; it means no recorded charging for that hour, not necessarily no charging anywhere.


In [8]:

hourly_chargeDemandTemp = grid.copy()
start_date = pd.Timestamp("2022-09-01 14:00:00")
end_date = pd.Timestamp("2023-12-31 23:00:00")

filtered_hourly_chargeDemandTemp = hourly_chargeDemandTemp[(hourly_chargeDemandTemp["local_time"] >= start_date) & (hourly_chargeDemandTemp["local_time"] <= end_date)] 
filtered_hourly_chargeDemandTemp = filtered_hourly_chargeDemandTemp[~hourly_chargeDemandTemp["local_time"].dt.normalize().isin(dst_dates)]


print(type(filtered_hourly_chargeDemandTemp))
print(filtered_hourly_chargeDemandTemp.shape)
print(filtered_hourly_chargeDemandTemp.head())

merged_hourly = pd.merge(filtered_hourly_chargeDemandTemp, hourly_homeCharging_demand, on="local_time", how="left")


print(merged_hourly["home_charging_kwh"].isna().sum())

merged_hourly["home_charging_kwh"] = merged_hourly["home_charging_kwh"].fillna(0)

print(merged_hourly["home_charging_kwh"].isna().sum())


display(merged_hourly.head())

<class 'pandas.DataFrame'>
(11602, 4)
      demand_mwh                  grid_utc  temp_f          local_time
5845         981 2022-09-01 20:00:00+00:00    88.9 2022-09-01 14:00:00
5846        1000 2022-09-01 21:00:00+00:00    88.2 2022-09-01 15:00:00
5847         997 2022-09-01 22:00:00+00:00    87.1 2022-09-01 16:00:00
5848        1009 2022-09-01 23:00:00+00:00    86.9 2022-09-01 17:00:00
5849        1002 2022-09-02 00:00:00+00:00    85.6 2022-09-01 18:00:00
51
0


C:\Users\Jason\AppData\Local\Temp\ipykernel_19424\730523494.py:6: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_hourly_chargeDemandTemp = filtered_hourly_chargeDemandTemp[~hourly_chargeDemandTemp["local_time"].dt.normalize().isin(dst_dates)]


,demand_mwh,grid_utc,temp_f,local_time,home_charging_kwh
0,981,2022-09-01 20:00:00+00:00,88.9,2022-09-01 14:00:00,5.287614
1,1000,2022-09-01 21:00:00+00:00,88.2,2022-09-01 15:00:00,3.563183
2,997,2022-09-01 22:00:00+00:00,87.1,2022-09-01 16:00:00,4.554365
3,1009,2022-09-01 23:00:00+00:00,86.9,2022-09-01 17:00:00,7.773969
4,1002,2022-09-02 00:00:00+00:00,85.6,2022-09-01 18:00:00,25.747746


### Restrict the study to overnight hours

Keep observations from 8:00 p.m. through 5:59 a.m. This focuses Hypothesis 1 on overnight grid demand and defines the same observation population for both candidate models.


In [9]:
overnight_observations = merged_hourly.copy()


hours = overnight_observations["local_time"].dt.hour

filtered_overnight_observations = overnight_observations[(hours >= 20) | (hours < 6)]

print("Overnight observations:", len(filtered_overnight_observations))

print(
    "Hours included:",
    sorted(filtered_overnight_observations["local_time"].dt.hour.unique())
)

display(filtered_overnight_observations.head())


Overnight observations: 4834
Hours included: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]


,demand_mwh,grid_utc,temp_f,local_time,home_charging_kwh
6,963,2022-09-02 02:00:00+00:00,83.8,2022-09-01 20:00:00,36.801465
7,905,2022-09-02 03:00:00+00:00,81.1,2022-09-01 21:00:00,33.986773
8,830,2022-09-02 04:00:00+00:00,79.8,2022-09-01 22:00:00,63.697259
9,747,2022-09-02 05:00:00+00:00,76.6,2022-09-01 23:00:00,72.290366
10,608,2022-09-02 06:00:00+00:00,75.1,2022-09-02 00:00:00,102.105306


### Group hours by charging intensity

Use quartiles of `home_charging_kwh` to compare low- and high-charging hours. This is descriptive R6 context, not the R7 model-selection criterion. If many hours have identical values, `qcut` can fail because quartile edges are not unique; check the distribution if that occurs.


In [10]:
quartile_observations = filtered_overnight_observations.copy()

home_kwh = quartile_observations["home_charging_kwh"]

charging_quartiles = pd.qcut(home_kwh, q=4, labels=["Q1", "Q2", "Q3", "Q4"])

quartile_observations["charging_quartile"] = charging_quartiles

print(quartile_observations["charging_quartile"].value_counts().sort_index())

display(
    quartile_observations[
        ["local_time", "home_charging_kwh", "charging_quartile"]
    ].head()
)

charging_quartile
Q1    1209
Q2    1208
Q3    1208
Q4    1209
Name: count, dtype: int64


,local_time,home_charging_kwh,charging_quartile
6,2022-09-01 20:00:00,36.801465,Q1
7,2022-09-01 21:00:00,33.986773,Q1
8,2022-09-01 22:00:00,63.697259,Q1
9,2022-09-01 23:00:00,72.290366,Q1
10,2022-09-02 00:00:00,102.105306,Q1


### Compare observed demand across charging quartiles

Calculate mean grid demand within each charging quartile and the difference between the highest and lowest quartiles. This describes an association without adjusting for temperature or time and does not establish causation.


In [11]:
q1 = quartile_observations.loc[quartile_observations["charging_quartile"] == "Q1"]
q2 = quartile_observations.loc[quartile_observations["charging_quartile"] == "Q2"]
q3 = quartile_observations.loc[quartile_observations["charging_quartile"] == "Q3"]
q4 = quartile_observations.loc[quartile_observations["charging_quartile"] == "Q4"]

display(q1["demand_mwh"].mean())
display(q2["demand_mwh"].mean())
display(q3["demand_mwh"].mean())
display(q4["demand_mwh"].mean())

difference = q4["demand_mwh"].mean() - q1["demand_mwh"].mean()

print(f"Q4 - Q1 difference: {difference:.2f} MW")


np.float64(622.3655913978495)

np.float64(657.25)

np.float64(702.5372516556291)

np.float64(697.671629445823)

Q4 - Q1 difference: 75.31 MW


### Train the charging-inclusive regression model

Create the prediction target (`demand_mwh`) and encode month and hour. Train ordinary least-squares linear regression using temperature, time indicators, and home charging. Linear regression is a reasonable initial candidate because the outcome is continuous and coefficients are interpretable. Sort chronologically and reserve the last 20% for validation to avoid random mixing of earlier and later hours.


In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd

# 1. Prepare regression data
regression_data = filtered_overnight_observations.copy()
regression_data = regression_data.sort_values("local_time")

# Add month and hour as categorical variables
regression_data["month"] = regression_data["local_time"].dt.month
regression_data["hour"] = regression_data["local_time"].dt.hour

# 2. Define input variables
X = regression_data[
    ["home_charging_kwh", "temp_f", "month", "hour"]
].copy()

# Encode month and hour as categorical features
X = pd.get_dummies(
    X,
    columns=["month", "hour"],
    drop_first=True,
    dtype=int
)

# Define the target variable
y = regression_data["demand_mwh"]

# 3. Split chronologically: 80% train, 20% validation
split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_val = X.iloc[split:]

y_train = y.iloc[:split]
y_val = y.iloc[split:]

# 4. Train linear regression
model = LinearRegression()
model.fit(X_train, y_train)

# 5. Generate predictions
train_predictions = model.predict(X_train)
val_predictions = model.predict(X_val)

# 6. Evaluate performance
print("Training R²:", r2_score(y_train, train_predictions))
print("Validation R²:", r2_score(y_val, val_predictions))

print("Training MAE:", mean_absolute_error(y_train, train_predictions))
print("Validation MAE:", mean_absolute_error(y_val, val_predictions))

# 7. Examine charging coefficient
coefficients = pd.Series(model.coef_, index=X.columns)

print(
    "\nHome charging coefficient:",
    coefficients["home_charging_kwh"]
)

print("Training observations:", len(X_train))
print("Validation observations:", len(X_val))

Training R²: 0.7809784579289706
Validation R²: 0.7486288550419883
Training MAE: 53.884460765067004
Validation MAE: 44.155850176306636

Home charging coefficient: 0.006092758963755289
Training observations: 3867
Validation observations: 967


### Build the matched baseline and compare models

Fit a second linear regression with the same target, estimator, preprocessing, training period, and validation period, removing **only** `home_charging_kwh`. This one-feature change isolates whether charging improves predictive performance. Compare validation R² (higher is better) and MAE (lower is better); these are model-selection metrics, not causal-effect estimates.


In [13]:
# Baseline model without home charging
X_baseline = X.drop(columns=["home_charging_kwh"])

# Use the exact same chronological split
X_base_train = X_baseline.iloc[:split]
X_base_val = X_baseline.iloc[split:]

baseline_model = LinearRegression()
baseline_model.fit(X_base_train, y_train)

baseline_predictions = baseline_model.predict(X_base_val)

# Compare validation performance
baseline_r2 = r2_score(y_val, baseline_predictions)
baseline_mae = mean_absolute_error(y_val, baseline_predictions)

charging_r2 = r2_score(y_val, val_predictions)
charging_mae = mean_absolute_error(y_val, val_predictions)

print("Without charging:")
print(f"Validation R²: {baseline_r2:.4f}")
print(f"Validation MAE: {baseline_mae:.2f} MW")

print("\nWith charging:")
print(f"Validation R²: {charging_r2:.4f}")
print(f"Validation MAE: {charging_mae:.2f} MW")

print(f"\nR² improvement: {charging_r2 - baseline_r2:.4f}")
print(f"MAE reduction: {baseline_mae - charging_mae:.2f} MW")

Without charging:
Validation R²: 0.7645
Validation MAE: 43.00 MW

With charging:
Validation R²: 0.7486
Validation MAE: 44.16 MW

R² improvement: -0.0158
MAE reduction: -1.15 MW


### Assess stability with time-series cross-validation

Within the earlier 80% training period, evaluate both feature sets on the same four expanding-window folds. Report fold-level R² and MAE, their means and standard deviations, and the train–validation gap. The final 20% holdout remains outside these folds. Note that the earlier cells already evaluate this holdout; a strict final-test-once workflow would choose the model from CV before opening the holdout.


In [14]:
from sklearn.model_selection import TimeSeriesSplit
import numpy as np

# Use the original training portion for cross-validation
tscv = TimeSeriesSplit(n_splits=4)

results = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), start=1):

    # Same chronological fold for both models
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    # Baseline: no home charging
    base = LinearRegression()
    base.fit(
        X_fold_train.drop(columns=["home_charging_kwh"]),
        y_fold_train
    )

    base_pred = base.predict(
        X_fold_val.drop(columns=["home_charging_kwh"])
    )

    # Full model: includes home charging
    full = LinearRegression()
    full.fit(X_fold_train, y_fold_train)

    full_pred = full.predict(X_fold_val)

    results.append({
        "fold": fold,
        "baseline_r2": r2_score(y_fold_val, base_pred),
        "full_r2": r2_score(y_fold_val, full_pred),
        "baseline_mae": mean_absolute_error(y_fold_val, base_pred),
        "full_mae": mean_absolute_error(y_fold_val, full_pred),
        "charging_coefficient": full.coef_[
            X_train.columns.get_loc("home_charging_kwh")
        ],
        "train_r2": r2_score(
            y_fold_train,
            full.predict(X_fold_train)
        )
    })

cv_results = pd.DataFrame(results)

cv_results["r2_improvement"] = (
    cv_results["full_r2"] - cv_results["baseline_r2"]
)

cv_results["mae_reduction"] = (
    cv_results["baseline_mae"] - cv_results["full_mae"]
)

cv_results["train_val_gap"] = (
    cv_results["train_r2"] - cv_results["full_r2"]
)

display(cv_results.round(4))

print("\nMean results:")
print(cv_results.mean(numeric_only=True).round(4))

print("\nStandard deviations:")
print(cv_results.std(numeric_only=True).round(4))

,fold,baseline_r2,full_r2,baseline_mae,full_mae,charging_coefficient,train_r2,r2_improvement,mae_reduction,train_val_gap
0,1,-1.4819,-2.2448,182.2225,213.8556,-0.0129,0.7761,-0.7630,-31.6331,3.0210
1,2,0.4101,0.4166,81.1189,77.8273,-0.0087,0.8827,0.0065,3.2916,0.4661
2,3,-0.6366,-0.7935,150.5529,157.5605,-0.0055,0.8817,-0.1570,-7.0077,1.6753
3,4,0.4498,0.4688,97.7397,95.9888,0.0013,0.8023,0.0190,1.7509,0.3335



Mean results:
fold                      2.5000
baseline_r2              -0.3146
full_r2                  -0.5382
baseline_mae            127.9085
full_mae                136.3081
charging_coefficient     -0.0064
train_r2                  0.8357
r2_improvement           -0.2236
mae_reduction            -8.3996
train_val_gap             1.3740
dtype: float64

Standard deviations:
fold                     1.2910
baseline_r2              0.9266
full_r2                  1.2785
baseline_mae            46.7696
full_mae                61.9434
charging_coefficient     0.0060
train_r2                 0.0547
r2_improvement           0.3684
mae_reduction           16.1395
train_val_gap            1.2530
dtype: float64


# R7 — Model Building and MLflow Tracking (Hypothesis 1)

**Question:** Does home EV charging add predictive information about overnight MCPA grid demand after accounting for temperature, month, and hour?

**Candidates:** Both use ordinary least-squares linear regression because the outcome, `demand_mwh`, is continuous. The baseline uses temperature and time controls; the second model adds only `home_charging_kwh`. This isolates the added predictive value of charging while keeping the estimator, chronological 80/20 split, preprocessing, and metrics fixed.

**Evaluation:** R² (higher is better) and MAE in MW (lower is better). The final 20% of hours is held out; four expanding-window folds within the earlier 80% provide stability information. These models test *predictive usefulness*, not causation.

**Prerequisites:** Run the existing cells above first, with the real Git LFS charging data downloaded. Install MLflow in the notebook environment if necessary: `python -m pip install mlflow`. Run this section after the prior cells; it records new MLflow runs from the same split and exports the run table. Do not interpret saved outputs as new executions.

In [15]:
# Verify chronological training and validation split

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))

print("Training first index:", X_train.index[0])
print("Training last index:", X_train.index[-1])

print("Validation first index:", X_val.index[0])
print("Validation last index:", X_val.index[-1])

# Check that observations do not overlap
assert X_train.index.intersection(X_val.index).empty

print("PASS: No overlapping observations.")

Training rows: 3867
Validation rows: 967
Training first index: 6
Training last index: 11460
Validation first index: 11461
Validation last index: 9393
PASS: No overlapping observations.


In [16]:
for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        if "local_time" in value.columns:
            print(name, value.shape)

grid (17520, 4)
hourly_homeCharging_demand (11566, 2)
hourly_chargeDemandTemp (17520, 4)
filtered_hourly_chargeDemandTemp (11602, 4)
merged_hourly (11602, 5)
overnight_observations (11602, 5)
filtered_overnight_observations (4834, 5)
quartile_observations (4834, 6)
q1 (1209, 6)
q2 (1208, 6)
q3 (1208, 6)
q4 (1209, 6)
regression_data (4834, 7)


In [17]:
# Verify chronological training and validation split

train_times = regression_data.loc[X_train.index, "local_time"]
val_times = regression_data.loc[X_val.index, "local_time"]

print("Training start:", train_times.min())
print("Training end:", train_times.max())

print("Validation start:", val_times.min())
print("Validation end:", val_times.max())

print("Training observations:", len(train_times))
print("Validation observations:", len(val_times))

# Verify there are no shared observations
assert X_train.index.intersection(X_val.index).empty

# Verify training dates come before validation dates
assert train_times.max() < val_times.min(), (
    "ERROR: Training and validation periods overlap!"
)

# Verify observations within each split are ordered by time
assert train_times.is_monotonic_increasing
assert val_times.is_monotonic_increasing

print("PASS: Chronological split verified.")

Training start: 2022-09-01 20:00:00
Training end: 2023-09-25 02:00:00
Validation start: 2023-09-25 03:00:00
Validation end: 2023-12-31 23:00:00
Training observations: 3867
Validation observations: 967
PASS: Chronological split verified.


### Track reproducible candidate runs in MLflow

Refit the two candidate models using the shared chronological split, then log model configuration, feature lists, training/validation metrics, fold metrics, and artifacts in MLflow. Export the run table with actual run IDs for the repository. Running this cell is required: saved notebook output alone is not proof of logged runs. The `mlruns` store and required CSV/artifacts must be preserved or shared according to your team’s repository policy.


In [ ]:

import os
import requests
import mlflow
import mlflow.sklearn
import pandas as pd

from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error

# Step 1: Configure the shared MLflow server and credentials.

MLFLOW_SERVER = "http://10.0.100.20:5000"
MLFLOW_USERNAME = "csc-4933-001-user03"
MLFLOW_PASSWORD = "sSuhuVoWzzqwMx5Y"

# Step 2: Connect to MLflow and verify access.

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_SERVER
os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_SERVER)

reply = requests.get(
    MLFLOW_SERVER + "/api/2.0/mlflow/experiments/search",
    params={"max_results": 1},
    auth=(MLFLOW_USERNAME, MLFLOW_PASSWORD),
    timeout=10
)

print("Server response:", reply.status_code)

if reply.status_code != 200:
    raise ConnectionError(
        f"MLflow connection failed: {reply.status_code}"
    )

EXPERIMENT = f"Group1-Climate-Energy-{MLFLOW_USERNAME}"
mlflow.set_experiment(EXPERIMENT)

print("Connected successfully!")
print("Experiment:", EXPERIMENT)

# Step 3: Verify training and validation data.

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert X_train.index.intersection(X_val.index).empty

# The original notebook must have sorted the data by time
# before creating the chronological training/validation split.

print("Training observations:", len(X_train))
print("Validation observations:", len(X_val))

# Step 4: Define candidate models.
# Both use the same regression algorithm and data split.
# Only the home charging feature changes.

candidate_features = {
    "R7_H1_baseline_weather_time": [
        c for c in X.columns
        if c != "home_charging_kwh"
    ],
    "R7_H1_add_home_charging": list(X.columns)
}

run_records = []

# Step 5: Train and evaluate each candidate model.

for run_name, features in candidate_features.items():

    model = LinearRegression()
    model.fit(X_train[features], y_train)

    train_pred = model.predict(X_train[features])
    val_pred = model.predict(X_val[features])

    metrics = {
        "train_r2": r2_score(y_train, train_pred),
        "train_mae_mw": mean_absolute_error(y_train, train_pred),
        "validation_r2": r2_score(y_val, val_pred),
        "validation_mae_mw": mean_absolute_error(y_val, val_pred)
    }

    metrics["train_validation_r2_gap"] = (
        metrics["train_r2"] - metrics["validation_r2"]
    )

    # Step 6: Four-fold time-series cross-validation.
    # Use expanding training windows within the training set.

    fold_results = []
    tscv = TimeSeriesSplit(n_splits=4)

    for fold, (train_idx, val_idx) in enumerate(
        tscv.split(X_train), start=1
    ):

        fold_model = LinearRegression()

        fold_model.fit(
            X_train.iloc[train_idx][features],
            y_train.iloc[train_idx]
        )

        fold_pred = fold_model.predict(
            X_train.iloc[val_idx][features]
        )

        fold_results.append({
            "fold": fold,
            "r2": r2_score(
                y_train.iloc[val_idx], fold_pred
            ),
            "mae_mw": mean_absolute_error(
                y_train.iloc[val_idx], fold_pred
            )
        })

    fold_df = pd.DataFrame(fold_results)

    metrics.update({
        "cv_r2_mean": fold_df["r2"].mean(),
        "cv_r2_std": fold_df["r2"].std(ddof=1),
        "cv_mae_mean_mw": fold_df["mae_mw"].mean(),
        "cv_mae_std_mw": fold_df["mae_mw"].std(ddof=1)
    })

    # Step 7: Log model parameters, metrics and artifacts.

    with mlflow.start_run(run_name=run_name) as run:

        mlflow.log_params({
            "estimator": "LinearRegression",
            "hypothesis": "H1",
            "split": "chronological_80_20",
            "cv": "TimeSeriesSplit_4",
            "feature_set": (
                "with_home_charging"
                if "home_charging_kwh" in features
                else "baseline"
            ),
            "features": ",".join(features),
            "n_train": len(y_train),
            "n_validation": len(y_val),
            "fit_intercept": True
        })

        mlflow.log_metrics({
            key: float(value)
            for key, value in metrics.items()
        })

        mlflow.sklearn.log_model(
            model,
            artifact_path="model"
        )

        fold_path = Path(f"{run_name}_cv_folds.csv")
        fold_df.to_csv(fold_path, index=False)

        mlflow.log_artifact(
            str(fold_path),
            artifact_path="validation"
        )

        mlflow.log_dict({
            "features": features,
            "target": "demand_mwh",
            "split": "chronological_80_20",
            "description": (
                "Same training and validation observations "
                "for both candidate models."
            )
        }, "feature_manifest.json")

        run_records.append({
            "run_name": run_name,
            "run_id": run.info.run_id,
            "features": ",".join(features),
            **metrics
        })

        print("Logged run:", run_name)
        print("Run ID:", run.info.run_id)

# Step 8: Export the model comparison table for R7.

run_table = pd.DataFrame(run_records)

run_table.to_csv(
    "R7_H1_mlflow_runs.csv",
    index=False
)

display(run_table.round(4))

print("MLflow server:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT)
print("Exported: R7_H1_mlflow_runs.csv")


SyntaxError: unterminated string literal (detected at line 17) (3206711343.py, line 17)

## Interpretation and repository checklist

The previously saved H1 results show that adding charging lowered validation R² from **0.7645** to **0.7486** and increased MAE from **43.00 MW** to **44.16 MW**. This is evidence that charging does not improve out-of-sample *prediction* in this linear specification, not proof that charging has no effect on demand. The new MLflow cell recomputes the comparison and must be executed to produce tracked runs.
